# WCCI Episodic Survival Plots

This notebook mirrors the A0 episodic-survival notebook, but it reads only the permanent WCCI `outputs/run_data` folders: `wcci_aib_24h`, `wcci_sparse16`, `wcci_hvg`, `wcci_baseline`, and `wcci_aib`. Edit the run-group cell to change which curves are compared.

In [ ]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from run_data import scan_run_data

RUN_DATA_ROOT = TASK_DIR / "outputs" / "run_data"
FIG_DIR = TASK_DIR / "outputs" / "wcci_metric_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

WCCI_GROUPS = [
    "wcci_aib_24h",
    "wcci_sparse16",
    "wcci_hvg",
    "wcci_baseline",
    "wcci_aib",
]
SEEDS = (0, 1, 2)
AGENTS = ["agent_0", "agent_1", "agent_2", "agent_3"]
SAVE_FIGURES = True
SHOW_FIGURES = True

print("Task dir:", TASK_DIR)
print("Run data root:", RUN_DATA_ROOT)
print("WCCI groups:", WCCI_GROUPS)


## Load Downloaded WCCI Histories

In [ ]:
def _path_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    path = Path(value).expanduser()
    return path if path.exists() else None


def read_history(row):
    parquet_path = _path_or_none(row.get("history_parquet"))
    csv_path = _path_or_none(row.get("history_csv"))
    if parquet_path is not None:
        history = pd.read_parquet(parquet_path)
    elif csv_path is not None:
        history = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"No local history file for {row.get('run_name')}")
    history = history.copy()
    history["run_name"] = row["run_name"]
    history["run_id"] = row["run_id"]
    history["download_group"] = row["download_group"]
    history["run_state"] = row.get("state")
    return history


run_index_all = scan_run_data(RUN_DATA_ROOT, include_legacy=False)
run_index = run_index_all[run_index_all["download_group"].isin(WCCI_GROUPS)].copy()
if run_index.empty:
    raise RuntimeError(f"No WCCI runs found under {RUN_DATA_ROOT}. Check that the downloaded groups exist.")

histories = []
for row in run_index.to_dict("records"):
    try:
        histories.append(read_history(row))
    except Exception as exc:
        print(f"Skipped {row.get('run_name')}: {exc}")

history_df = pd.concat(histories, ignore_index=True, sort=False) if histories else pd.DataFrame()
if history_df.empty:
    raise RuntimeError("No WCCI histories could be loaded.")

coverage = (
    run_index.assign(
        family=lambda df: df["run_name"].map(lambda name: re.sub(r"_s\\d+$", "", str(name))),
        seed=lambda df: df["run_name"].map(lambda name: int(re.search(r"_s(\\d+)$", str(name)).group(1)) if re.search(r"_s(\\d+)$", str(name)) else np.nan),
    )
    .groupby(["download_group", "family", "state"], dropna=False, as_index=False)
    .agg(
        runs=("run_name", "count"),
        seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        rows_min=("rows", "min"),
        rows_max=("rows", "max"),
    )
    .sort_values(["download_group", "family", "state"])
)
print(f"Loaded {len(run_index)} runs and {len(history_df):,} history rows.")
display(coverage)


## Labels and Run Helpers

In [ ]:
def safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-") or "plot"


def save_figure(fig, name):
    if fig is None or not SAVE_FIGURES:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print("Saved:", path)
    return path


def seed_from_run(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else np.nan


def family_from_run(run_name):
    return re.sub(r"_s\d+$", "", str(run_name))


WCCI_LABELS = {
    "wcci_reduced_mlp_baseline": "reduced MLP baseline 24h",
    "wcci_reduced_mlp_baseline_72x576_s0": "baseline 60M old name",
    "wcci_reduced_mlp_baseline_72x576": "reduced MLP baseline 60M",
    "wcci_reduced_mlp_baseline_72x576_lr20m_f010": "reduced MLP baseline lr20m f0.10",
    "wcci_mlp_baseline_72x576_20M": "MLP baseline 20M",
    "wcci_mlp_baseline_72x576_60M": "MLP baseline 60M",
    "wcci_hvg_01_eval_rho090_72x576": "global rho heuristic 0.90",
    "wcci_hvg_04_eval_local_rho090_72x576": "local rho heuristic 0.90",
    "wcci_hvg_02_gate_final_map_72x576": "gate final-action MAP",
    "wcci_hvg_03_gate_hierarchical_72x576": "gate hierarchical greedy",
    "wcci_sparse16_flat_p003_72x576": "Sparse16 flat p0.003",
    "wcci_aib_00_flat_local_t020_72x576": "AIB flat local t0.20",
    "wcci_aib_00_flat_local_t020_72x576_lr20m_f010": "AIB flat local t0.20 lr20m f0.10",
    "wcci_aib_01_flat_local_t010_72x576": "AIB flat local t0.10",
    "wcci_aib_02_flat_local_t035_72x576": "AIB flat local t0.35",
    "wcci_aib_03_gate_hgreedy_sep_local_t020_72x576": "AIB gate h-greedy t0.20",
    "wcci_aib_04_flat_nonidle_t020_72x576": "AIB flat non-idle t0.20",
    "wcci_aib_01_flat_local_t010_topo003_72x576": "AIB t0.10 topo0.003",
    "wcci_aib_01_flat_local_t010_topo010_72x576": "AIB t0.10 topo0.010",
}

WCCI_COLOR = {
    "MLP baseline 20M": "#1f77b4",
    "MLP baseline 60M": "#4e79a7",
    "reduced MLP baseline 60M": "#1f77b4",
    "reduced MLP baseline lr20m f0.10": "#17becf",
    "global rho heuristic 0.90": "#ff7f0e",
    "local rho heuristic 0.90": "#2ca02c",
    "gate final-action MAP": "#9467bd",
    "gate hierarchical greedy": "#d62728",
    "Sparse16 flat p0.003": "#8c564b",
    "AIB flat local t0.20": "#ff7f0e",
    "AIB flat local t0.20 lr20m f0.10": "#bcbd22",
    "AIB flat local t0.10": "#2ca02c",
    "AIB flat local t0.35": "#d62728",
    "AIB gate h-greedy t0.20": "#8c564b",
    "AIB flat non-idle t0.20": "#9467bd",
    "AIB t0.10 topo0.003": "#e377c2",
    "AIB t0.10 topo0.010": "#7f7f7f",
}


def pretty_label(family):
    return WCCI_LABELS.get(str(family), str(family).replace("wcci_", "").replace("_", " "))


def seeded(prefix, seeds=SEEDS):
    return [f"{prefix}_s{seed}" for seed in seeds]


def available_run_names(history=None):
    data = history_df if history is None else history
    return set(data["run_name"].dropna().astype(str).unique())


def report_missing_runs(run_groups, history=None):
    available = available_run_names(history)
    rows = []
    for label, runs in run_groups.items():
        missing = [run for run in runs if run not in available]
        rows.append({"curve": label, "expected": len(runs), "available": len(runs) - len(missing), "missing": missing})
    coverage = pd.DataFrame(rows)
    display(coverage)
    return coverage


def infer_family_table():
    rows = []
    for run_name in sorted(history_df["run_name"].dropna().astype(str).unique()):
        rows.append({
            "download_group": history_df.loc[history_df["run_name"].eq(run_name), "download_group"].iloc[0],
            "run_name": run_name,
            "family": family_from_run(run_name),
            "label": pretty_label(family_from_run(run_name)),
            "seed": seed_from_run(run_name),
        })
    return pd.DataFrame(rows)


family_table = infer_family_table()
display(family_table.sort_values(["download_group", "family", "seed"]))


## Survival Helpers

In [ ]:
SURVIVAL_METRICS = {
    "test": ["test/episodic_survival", "test/charts/episodic_survival"],
    "train_eval": ["train_eval/episodic_survival", "train_eval/charts/episodic_survival"],
    "legacy": ["charts/episodic_survival"],
}


def first_existing_metric(candidates, data=None):
    data = history_df if data is None else data
    for metric in candidates:
        if metric in data.columns:
            return metric
    raise KeyError(f"None of these metrics are available: {candidates}")


def _survival_scale(values):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0
    return 100.0 if max_value <= 1.5 else 1.0


def survival_long(run_groups, *, split="test", smooth=5, history=None):
    data = history_df if history is None else history
    metric = first_existing_metric(SURVIVAL_METRICS[split], data)
    rows = []
    for label, runs in run_groups.items():
        for run_name in runs:
            run_data = data[data["run_name"].astype(str).eq(str(run_name))]
            if run_data.empty:
                continue
            cols = ["run_name", "run_id", "download_group", "_step", metric]
            cols = [col for col in cols if col in run_data.columns]
            curve = run_data[cols].copy().dropna(subset=[metric, "_step"])
            if curve.empty:
                continue
            scale = _survival_scale(curve[metric])
            curve["value"] = pd.to_numeric(curve[metric], errors="coerce") * scale
            curve["step_millions"] = pd.to_numeric(curve["_step"], errors="coerce") / 1_000_000.0
            curve["curve"] = label
            curve["seed"] = seed_from_run(run_name)
            curve["family"] = family_from_run(run_name)
            curve["metric"] = metric
            curve = curve.sort_values("_step")
            if smooth and smooth > 1:
                curve["value_smooth"] = curve["value"].rolling(int(smooth), min_periods=1).mean()
            else:
                curve["value_smooth"] = curve["value"]
            rows.append(curve)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def mean_survival_curves(long_df):
    if long_df.empty:
        return pd.DataFrame()
    return (
        long_df
        .groupby(["curve", "_step", "step_millions"], as_index=False, observed=True)
        .agg(
            mean_survival=("value_smooth", "mean"),
            std_survival=("value_smooth", "std"),
            min_survival=("value_smooth", "min"),
            max_survival=("value_smooth", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )


def plot_survival_groups(run_groups, *, title, split="test", smooth=5, y_range=(0, 105), show_members=True, show_std=True, save_name=None, history=None):
    long_df = survival_long(run_groups, split=split, smooth=smooth, history=history)
    if long_df.empty:
        print(f"No survival data for {title}")
        return None, long_df, pd.DataFrame()
    mean_df = mean_survival_curves(long_df)
    fig = go.Figure()
    for label, runs in run_groups.items():
        color = WCCI_COLOR.get(label, None)
        curve_data = mean_df[mean_df["curve"].eq(label)].sort_values("step_millions")
        if curve_data.empty:
            continue
        if show_members:
            for run_name, member in long_df[long_df["curve"].eq(label)].groupby("run_name", sort=False):
                member = member.sort_values("step_millions")
                fig.add_trace(go.Scatter(
                    x=member["step_millions"],
                    y=member["value_smooth"],
                    mode="lines",
                    name=f"{label} seed {seed_from_run(run_name)}",
                    legendgroup=label,
                    showlegend=False,
                    line={"color": color, "width": 1.2},
                    opacity=0.18,
                    hovertemplate=(
                        f"<b>{label}</b><br>run={run_name}<br>"
                        "step=%{x:.2f}M<br>survival=%{y:.2f}%<extra></extra>"
                    ),
                ))
        if show_std and curve_data["n_seeds"].max() > 1:
            std = curve_data["std_survival"].fillna(0.0)
            fig.add_trace(go.Scatter(
                x=pd.concat([curve_data["step_millions"], curve_data["step_millions"].iloc[::-1]]),
                y=pd.concat([curve_data["mean_survival"] + std, (curve_data["mean_survival"] - std).iloc[::-1]]),
                fill="toself",
                fillcolor=color if color else "rgba(31,119,180,0.14)",
                line={"color": "rgba(255,255,255,0)"},
                opacity=0.12,
                hoverinfo="skip",
                showlegend=False,
                legendgroup=label,
            ))
        fig.add_trace(go.Scatter(
            x=curve_data["step_millions"],
            y=curve_data["mean_survival"],
            mode="lines",
            name=label,
            legendgroup=label,
            line={"color": color, "width": 3.5},
            customdata=np.stack([curve_data["n_seeds"], curve_data["seeds"].astype(str)], axis=-1),
            hovertemplate=(
                f"<b>{label}</b><br>step=%{{x:.2f}}M<br>"
                "mean survival=%{y:.2f}%<br>n_seeds=%{customdata[0]}<br>seeds=%{customdata[1]}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1350,
        height=650,
        xaxis_title="steps (M)",
        yaxis_title=f"{split} episodic survival (%)",
        hovermode="x unified",
        legend=dict(
            x=0.985,
            y=0.035,
            xanchor="right",
            yanchor="bottom",
            bgcolor="rgba(255,255,255,0.88)",
            bordercolor="rgba(80,80,80,0.25)",
            borderwidth=1,
            font=dict(size=14),
        ),
        margin={"l": 80, "r": 40, "t": 90, "b": 70},
    )
    if y_range is not None:
        fig.update_yaxes(range=list(y_range))
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, long_df, mean_df


def latest_survival_summary(run_groups, *, split="test", last_n=5, history=None):
    long_df = survival_long(run_groups, split=split, smooth=1, history=history)
    if long_df.empty:
        return pd.DataFrame()
    per_run = (
        long_df.sort_values(["run_name", "_step"])
        .groupby(["curve", "run_name", "seed"], as_index=False, observed=True)
        .tail(int(last_n))
        .groupby(["curve", "run_name", "seed"], as_index=False, observed=True)
        .agg(final_survival=("value", "mean"), final_step_m=("step_millions", "max"))
    )
    summary = (
        per_run.groupby("curve", as_index=False, observed=True)
        .agg(
            mean_survival=("final_survival", "mean"),
            std_survival=("final_survival", "std"),
            min_survival=("final_survival", "min"),
            max_survival=("final_survival", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            max_step_m=("final_step_m", "max"),
        )
        .sort_values("mean_survival", ascending=False)
    )
    return summary


## Editable WCCI Run Groups

In [ ]:
# Edit this cell to choose which WCCI runs appear in each figure.
# The notebook only uses downloaded permanent run_data, no W&B API calls.
BASELINE_FOR_COMPARISONS = "wcci_reduced_mlp_baseline_72x576_lr20m_f010"
BASELINE_LABEL = "reduced MLP baseline lr20m f0.10"

WCCI_BASELINE_RUNS = {
    "MLP baseline 20M": seeded("wcci_mlp_baseline_72x576_20M"),
    "MLP baseline 60M": seeded("wcci_mlp_baseline_72x576_60M"),
    "reduced MLP baseline lr20m f0.10": seeded("wcci_reduced_mlp_baseline_72x576_lr20m_f010"),
}

WCCI_HVG_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "global rho heuristic 0.90": seeded("wcci_hvg_01_eval_rho090_72x576"),
    "local rho heuristic 0.90": seeded("wcci_hvg_04_eval_local_rho090_72x576"),
    "gate final-action MAP": seeded("wcci_hvg_02_gate_final_map_72x576"),
    "gate hierarchical greedy": seeded("wcci_hvg_03_gate_hierarchical_72x576"),
}

WCCI_SPARSE16_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "Sparse16 flat p0.003": seeded("wcci_sparse16_flat_p003_72x576"),
}

WCCI_AIB_24H_RUNS = {
    "reduced MLP baseline 24h": seeded("wcci_reduced_mlp_baseline"),
    "AIB flat local t0.20": seeded("wcci_aib_00_flat_local_t020_72x576"),
    "AIB flat local t0.10": seeded("wcci_aib_01_flat_local_t010_72x576"),
    "AIB flat local t0.35": seeded("wcci_aib_02_flat_local_t035_72x576"),
}

WCCI_AIB_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "AIB flat local t0.20": seeded("wcci_aib_00_flat_local_t020_72x576"),
    "AIB flat local t0.20 lr20m f0.10": seeded("wcci_aib_00_flat_local_t020_72x576_lr20m_f010"),
    "AIB flat local t0.10": seeded("wcci_aib_01_flat_local_t010_72x576"),
    "AIB t0.10 topo0.003": seeded("wcci_aib_01_flat_local_t010_topo003_72x576"),
    "AIB t0.10 topo0.010": seeded("wcci_aib_01_flat_local_t010_topo010_72x576"),
}

ALL_COMPARISONS = {
    "WCCI baselines": WCCI_BASELINE_RUNS,
    "WCCI HVG": WCCI_HVG_RUNS,
    "WCCI Sparse16": WCCI_SPARSE16_RUNS,
    "WCCI AIB 24h": WCCI_AIB_24H_RUNS,
    "WCCI AIB": WCCI_AIB_RUNS,
}

# Some AIB run names exist in both wcci_aib_24h and wcci_aib.
# Always filter by the downloaded source folder as well as run name.
COMPARISON_SOURCE_GROUPS = {
    "WCCI baselines": ["wcci_baseline"],
    "WCCI HVG": ["wcci_baseline", "wcci_hvg"],
    "WCCI Sparse16": ["wcci_baseline", "wcci_sparse16"],
    "WCCI AIB 24h": ["wcci_aib_24h"],
    "WCCI AIB": ["wcci_baseline", "wcci_aib"],
}


def source_history(comparison_name):
    sources = COMPARISON_SOURCE_GROUPS.get(comparison_name, WCCI_GROUPS)
    return history_df[history_df["download_group"].isin(sources)].copy()


def source_rows(rows, comparison_name):
    if rows is None or rows.empty or "download_group" not in rows.columns:
        return rows
    sources = COMPARISON_SOURCE_GROUPS.get(comparison_name, WCCI_GROUPS)
    return rows[rows["download_group"].isin(sources)].copy()


for name, groups in ALL_COMPARISONS.items():
    print(f"\n{name}")
    report_missing_runs(groups, source_history(name))


## WCCI Baselines

In [ ]:
fig_wcci_baselines, wcci_baseline_long, wcci_baseline_mean = plot_survival_groups(
    WCCI_BASELINE_RUNS,
    title="WCCI baselines: test episodic survival",
    split="test",
    smooth=5,
    save_name="wcci_baselines_test_survival",
    history=source_history("WCCI baselines"),
)
fig_wcci_baselines

## WCCI Heuristic vs Gate

In [ ]:
fig_wcci_hvg, wcci_hvg_long, wcci_hvg_mean = plot_survival_groups(
    WCCI_HVG_RUNS,
    title="WCCI heuristic vs gate: test episodic survival",
    split="test",
    smooth=5,
    save_name="wcci_hvg_test_survival",
    history=source_history("WCCI HVG"),
)
fig_wcci_hvg

## WCCI Sparse16

In [ ]:
fig_wcci_sparse16, wcci_sparse16_long, wcci_sparse16_mean = plot_survival_groups(
    WCCI_SPARSE16_RUNS,
    title="WCCI Sparse16: test episodic survival",
    split="test",
    smooth=5,
    save_name="wcci_sparse16_test_survival",
    history=source_history("WCCI Sparse16"),
)
fig_wcci_sparse16

## WCCI AIB 24h

In [ ]:
fig_wcci_aib_24h, wcci_aib_24h_long, wcci_aib_24h_mean = plot_survival_groups(
    WCCI_AIB_24H_RUNS,
    title="WCCI AIB 24h: test episodic survival",
    split="test",
    smooth=5,
    save_name="wcci_aib_24h_test_survival",
    history=source_history("WCCI AIB 24h"),
)
fig_wcci_aib_24h

## WCCI AIB

In [ ]:
fig_wcci_aib, wcci_aib_long, wcci_aib_mean = plot_survival_groups(
    WCCI_AIB_RUNS,
    title="WCCI AIB: test episodic survival",
    split="test",
    smooth=5,
    save_name="wcci_aib_test_survival",
    history=source_history("WCCI AIB"),
)
fig_wcci_aib

## Final Survival Summary

In [ ]:
summary_tables = []
for comparison_name, groups in ALL_COMPARISONS.items():
    summary = latest_survival_summary(groups, split="test", last_n=5, history=source_history(comparison_name))
    summary.insert(0, "comparison", comparison_name)
    summary_tables.append(summary)
final_survival_summary = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
display(final_survival_summary)